In [1]:
import pandas as pd
import glob
import os
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from copy import deepcopy

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from collections import Counter
from torchvision import transforms as T
from torch.utils.data import DataLoader

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import utils as vutils
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

%env CUDA_VISIBLE_DEVICES=1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env: CUDA_VISIBLE_DEVICES=1


#### Dataset preparing

In [ ]:
dataset_root = '/Users/anmilka/Downloads/celeba/img_align_celeba/img_align_celeba'
# test_img_path = f'{dataset_root}/Abel_Pacheco/Abel_Pacheco_0001.jpg'
test_img_path = f'{dataset_root}/000008.jpg'
# test_img_path = f'{dataset_root}/John_Bolton/John_Bolton_0016.jpg'

pil_img = Image.open(test_img_path)
print(pil_img.size)
pil_img

In [ ]:
from ultralytics import YOLO
import torch

device, fp16 = ('cuda', True) if torch.cuda.is_available() else ('cpu', False)

detector_model = YOLO('yolov8n-face.pt')
detector_model.to(device)
res = detector_model.predict(source=test_img_path,
                              show=False,
                              save=False,
                              conf=0.4,
                              save_txt=False,
                              save_crop=False,
                              verbose=False,
                              device=device,
                              half=fp16,
                              )[0]

In [ ]:
faces = []
for ind in range(len(res.boxes)):
    box_points = res.boxes.xyxy[ind].cpu().numpy().astype(int)
    keypoints = res.keypoints[ind].data.cpu().numpy()[0][:, :2].astype(int)
    faces.append(
        {
            "conf": res.boxes.conf[ind].cpu().item(),
            "box_points": box_points.tolist(),
            "key_points": keypoints.tolist(),
        }
    )
faces

[{'conf': 0.8591823577880859,
  'box_points': [55, 61, 133, 179],
  'key_points': [[71, 112], [106, 112], [82, 137], [74, 153], [103, 153]]}]

In [ ]:
import cv2

def draw_detection_results(image, face_info):
    conf = face_info['conf']
    box_points = face_info['box_points']
    key_points = face_info['key_points']

    x0, y0, x1, y1 = tuple(box_points)
    color = (0, 0, 255)
    text = f'{conf:.2f}'
    font = cv2.FONT_HERSHEY_SIMPLEX
    text_size, _ = cv2.getTextSize(text, font, 0.5, 2)
    text_x = int((x1 + x0 - text_size[0]) // 2)
    text_y = int(y0 - text_size[1])
    cv2.rectangle(image, (int(x0), int(y0)), (int(x1), int(y1)), color, 2)
    cv2.putText(image, text, (text_x, text_y), font, 0.5, color, 2)

    key_points_colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (0, 255, 255), (128, 0, 128)]
    radius = 3
    for point, cur_color in zip(key_points, key_points_colors):
        cv2.circle(image, (int(point[0]), int(point[1])), radius, cur_color, -1)

In [ ]:
from PIL import Image
img = cv2.imread(test_img_path)

for face_info in faces:
    draw_detection_results(img, face_info)

print(img.shape)
display(Image.fromarray(img[:,:,::-1]))

(218, 178, 3)


In [ ]:
def get_central_of_face(face_info):
    left_eye, right_eye, nose = face_info['key_points'][:3]
    center_x = (left_eye[0] + right_eye[0] + nose[0]) // 3
    center_y = (left_eye[1] + right_eye[1] + nose[1]) // 3
    face_center_point = center_x, center_y
    return face_center_point

def find_most_central_face(img, faces):
    image_height, image_width, C = img.shape

    min_distance = float('inf')
    most_central_face = None

    for face_info in faces:
        center_x, center_y = get_central_of_face(face_info)
        distance = ((center_x - image_width/2)**2 + (center_y - image_height/2)**2)**0.5

        if distance < min_distance:
            min_distance = distance
            most_central_face = face_info

    return most_central_face

face_info = find_most_central_face(img, faces)
center_x, center_y = face_center_point = get_central_of_face(face_info)

cv2.circle(img, (int(center_x), int(center_y)), 3, (0, 0, 0), -1)

display(Image.fromarray(img[:,:,::-1]))

In [ ]:
import numpy as np

def crop_with_padding(image, new_bbox):
    return np.array(Image.fromarray(image).crop(new_bbox))

box_points = face_info['box_points']

max_sz = max(
        abs(box_points[0] - center_x),
        abs(box_points[1] - center_y),
        abs(box_points[2] - center_x),
        abs(box_points[3] - center_y),
    )

new_bbox = (face_center_point[0] - max_sz, face_center_point[1] - max_sz,
            face_center_point[0] + max_sz, face_center_point[1] + max_sz)
croped_img = crop_with_padding(img, new_bbox)

display(Image.fromarray(croped_img[:,:,::-1]))

In [ ]:
import os
import cv2
from tqdm import tqdm

IMG_SIZE = (64, 64)

def preprocess_image(image_path):
    res = detector_model.predict(source=image_path,
                            show=False,
                            save=False,
                            conf=0.4,
                            save_txt=False,
                            save_crop=False,
                            verbose=False,
                            device=device,
                            half=fp16,
                            )[0]
    faces = []
    for ind in range(len(res.boxes)):
        box_points = res.boxes.xyxy[ind].cpu().numpy().astype(int)
        keypoints = res.keypoints[ind].data.cpu().numpy()[0][:, :2].astype(int)
        faces.append(
            {
                "conf": res.boxes.conf[ind].cpu().item(),
                "box_points": box_points.tolist(),
                "key_points": keypoints.tolist(),
            }
        )

    if len(faces) == 0:
        print(f"Error: found no faces on image: {image_path}!!!")
        return None

    img = cv2.imread(image_path)
    face_info = find_most_central_face(img, faces)
    center_x, center_y = face_center_point = get_central_of_face(face_info)
    box_points = face_info['box_points']

    max_sz = max(
            abs(box_points[0] - center_x),
            abs(box_points[1] - center_y),
            abs(box_points[2] - center_x),
            abs(box_points[3] - center_y),
        )

    new_bbox = (face_center_point[0] - max_sz, face_center_point[1] - max_sz,
                face_center_point[0] + max_sz, face_center_point[1] + max_sz)
    croped_img = crop_with_padding(img, new_bbox)
    preprocessed_image = cv2.resize(croped_img, IMG_SIZE)
    return preprocessed_image


LFW_yolo_root = '/Users/anmilka/Downloads/celeba_cropped'
os.makedirs(LFW_yolo_root, exist_ok=True)


for i, image_file in enumerate(os.listdir(dataset_root)):
    try:
        #print(image_file)
        image_path = os.path.join(dataset_root, image_file)

        preprocessed_image = preprocess_image(image_path)
        output_image_path = os.path.join(LFW_yolo_root, image_file)
        cv2.imwrite(output_image_path, preprocessed_image)
    except:
        continue

    if i % 500 == 0:
        print(i)
    if i==5000:
        break

0
Error: found no faces on image: /Users/anmilka/Downloads/celeba/img_align_celeba/img_align_celeba/145340.jpg!!!
500
Error: found no faces on image: /Users/anmilka/Downloads/celeba/img_align_celeba/img_align_celeba/043080.jpg!!!
1000
1500
2000
2500
Error: found no faces on image: /Users/anmilka/Downloads/celeba/img_align_celeba/img_align_celeba/129701.jpg!!!
3000
3500
4000
4500
Error: found no faces on image: /Users/anmilka/Downloads/celeba/img_align_celeba/img_align_celeba/108320.jpg!!!
5000


#### Dataloaders

In [2]:
import torchvision.transforms as transforms

LFW_yolo_root = '/home/anmilka/different/celeba_cropped'

# Определение трансформаций
transforms_=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

class ImageDataset(Dataset):
    def __init__(self, paths, transform):
        self.transform = transform
        self.paths = sorted(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        try:
            image = Image.open(self.paths[index])
        except:
            print("error in open", self.paths[index])

        try:
            image_tensor = self.transform(image)
        except Exception as e:
            print("error in transform", self.paths[index], e)
            return

        return image_tensor

paths = glob.glob(f"{LFW_yolo_root}/*.jpg")
print(paths[0])
train_paths, test_paths = train_test_split(paths, shuffle=True, test_size=0.05, random_state=25)
print(len(train_paths), len(test_paths))

train_dataset = ImageDataset(paths=sorted(train_paths),
                            transform=transforms_)
                            
test_dataset = ImageDataset(paths=sorted(test_paths),
                            transform=transforms_)

print(len(train_dataset), len(test_dataset))

batch_size = 64
num_workers = 2

train_loader = DataLoader(train_dataset,
                            batch_size=batch_size,
                            shuffle=True,
                            num_workers=num_workers,
                            pin_memory=True,
                            drop_last = True)

val_loader = DataLoader(test_dataset,
                            batch_size=batch_size,
                            shuffle=False,
                            num_workers=num_workers,
                            pin_memory=True)

print(len(train_loader), len(val_loader))

/home/anmilka/different/celeba_cropped/104255.jpg
18977 999
18977 999
296 16


In [3]:
def show(img):
    img = img.detach().cpu()
    img = (np.transpose((img / 2 + 0.5).clamp(0, 1).numpy(), (1, 2, 0)) * 255).astype(np.uint8)
    display(Image.fromarray(img))

def plot_training_metrics(metrics):
    fig, axs = plt.subplots(1, 2, figsize=(28, 7))

    plt.subplot(1, 3, 1)
    plt.plot(np.arange(len(metrics["train_kl"])), metrics["train_kl"], label='Train kl')
    plt.plot(np.arange(len(metrics["train_mse"])), metrics["train_mse"], label='Train mse')
    plt.title('Train Loss')
    plt.xlabel('Step')
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(np.arange(len(metrics["val_kl"])), metrics["val_kl"], label='Val kl')
    plt.plot(np.arange(len(metrics["val_mse"])), metrics["val_mse"], label='Val mse')
    plt.title('Val Loss')
    plt.xlabel('Epoch')
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(np.arange(len(metrics["val_fid"])), metrics["val_fid"], label='Val FID')
    plt.plot(np.arange(len(metrics["is_value"])), metrics["is_value"], label='IS val')
    plt.plot(np.arange(len(metrics["is_std"])), metrics["is_std"], label='IS std')
    plt.title('Val Loss')
    plt.xlabel('Epoch')
    plt.legend()

    plt.tight_layout()
    plt.show()


def train_model(model, train_loader, val_loader, optimizer, device, num_epochs=25, log_steps=20):
    model.to(device)
    metrics = {}

    metrics["train_kl"] = []
    metrics["train_mse"] = []

    metrics["val_kl"] = []
    metrics["val_mse"] = []

    metrics["val_fid"] = []
    metrics["is_value"] = []
    metrics["is_std"] = []

    step = 0

    for epoch in range(num_epochs):
        model.train()
        for inputs in train_loader:
            inputs = inputs.to(device)
            optimizer.zero_grad()
            preds, mu, std = model(inputs)
            loss, recons_loss, kld_loss = compute_loss(inputs, preds, mu, std)
            if step % log_steps == 0:
                metrics["train_kl"].append(kld_loss.item())
                metrics["train_mse"].append(recons_loss.item())

                print("Train preds:")
                show(vutils.make_grid(preds[:4]))        

                print(f'Epoch {epoch}/{num_epochs - 1}, Step {step}, Train KL Loss: {metrics["train_kl"][-1]:.4f}, Train MSE Loss: {metrics["train_mse"][-1]:.4f}')

            loss.backward()
            optimizer.step()
            step +=1

        #scheduler.step()

        # Validation phase
        model.eval()

        with torch.no_grad():
            for inputs in val_loader:
                inputs = inputs.to(device)
                preds, mu, std = model(inputs)
                loss, recons_loss, kld_loss = compute_loss(inputs, preds, mu, std)

                fid_metric = FrechetInceptionDistance(feature=64, normalize=True).to(device)
                is_metric = InceptionScore(feature=64, normalize=True).to(device)

                fid_metric.update(inputs, real=True)
                fid_metric.update(preds, real=False)
                is_metric.update(preds)

                fid_value = fid_metric.compute()
                is_value, is_std = is_metric.compute()

                metrics["val_fid"].append(fid_value.item())
                metrics["is_value"].append(is_value.item())
                metrics["is_std"].append(is_std.item())

                metrics["val_kl"].append(kld_loss.item())
                metrics["val_mse"].append(recons_loss.item())

            print("Val preds:")
            show(vutils.make_grid(preds[:4]))        
            print(f'Epoch {epoch}/{num_epochs - 1}, Step {step}, Val KL Loss: {metrics["val_kl"][-1]:.4f}, Val MSE Loss: {metrics["val_mse"][-1]:.4f},\
 Val FID: {metrics["val_fid"][-1]:.4f}, IS val: {metrics["is_value"][-1]:.4f}, IS std: {metrics["is_std"][-1]:.4f}')

    return model, metrics

In [4]:
import torch
from torch import nn
import math
import torch.nn.functional as F
from torchsummary import summary

hidden_sizes = [256, 128, 64, 32, 16, 8, 3]

class Decoder(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.fc = nn.Linear(latent_dim, hidden_sizes[0] * 1 * 1)
        self.layers = nn.Sequential(
            nn.ConvTranspose2d(hidden_sizes[0], hidden_sizes[1], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[1]),
            nn.LeakyReLU(),

            nn.ConvTranspose2d(hidden_sizes[1], hidden_sizes[2], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[2]),
            nn.LeakyReLU(),

            nn.ConvTranspose2d(hidden_sizes[2], hidden_sizes[3], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[3]),
            nn.LeakyReLU(),

            nn.ConvTranspose2d(hidden_sizes[3], hidden_sizes[4], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[4]),
            nn.LeakyReLU(),

            nn.ConvTranspose2d(hidden_sizes[4], hidden_sizes[5], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[5]),
            nn.LeakyReLU(),

            nn.ConvTranspose2d(hidden_sizes[5], hidden_sizes[6], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[6]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[6], out_channels=3, kernel_size=3, padding= 1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.fc(x)
        x = x.reshape(-1, hidden_sizes[0], 1, 1)
        x = self.layers(x)
        return x

class Encoder(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(hidden_sizes[-1], hidden_sizes[-2], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_sizes[-2]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[-2], hidden_sizes[-3], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_sizes[-3]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[-3], hidden_sizes[-4], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_sizes[-4]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[-4], hidden_sizes[-5], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_sizes[-5]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[-5], hidden_sizes[-6], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_sizes[-6]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[-6], hidden_sizes[-7], kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU()
        )
        self.fc_mu = nn.Linear(hidden_sizes[0], latent_dim)
        self.fc_var = nn.Linear(hidden_sizes[0], latent_dim)

    def forward(self, x):
        x = self.layers(x)
        x = torch.flatten(x, 1)
    
        mu = self.fc_mu(x)
        std = self.fc_var(x)

        return mu, std

def reparametrization_trick(mu, std):
        std = torch.exp(std)
        eps = torch.randn_like(std)
        z = eps * std + mu

        return z

class VAE(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.encoder = Encoder(latent_dim=latent_dim)
        self.decoder = Decoder(latent_dim=latent_dim)

    def forward(self, x):
        mu, std = self.encoder(x)        

        # reparametrization trick
        z = reparametrization_trick(mu, std)

        x_new = self.decoder(z)
    
        return x_new, mu, std
# m = Decoder(100)
# x = torch.randint(3, 10, (1, 100), dtype=torch.float)
# m(x)
# summary(m, input_size=(1, 100), batch_size=16)


# m = Encoder(100)
# x = torch.randint(3, 10, (1, 3, 32, 32), dtype=torch.float)
# m(x)
# summary(m, input_size=(3, 32, 32), batch_size=16)

m = VAE(100).to(device)
x = torch.randint(3, 10, (1, 3, 64, 64), dtype=torch.float).to(device)
m(x)
summary(m, input_size=(3, 64, 64), batch_size=7)


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1             [7, 8, 32, 32]             224
       BatchNorm2d-2             [7, 8, 32, 32]              16
         LeakyReLU-3             [7, 8, 32, 32]               0
            Conv2d-4            [7, 16, 16, 16]           1,168
       BatchNorm2d-5            [7, 16, 16, 16]              32
         LeakyReLU-6            [7, 16, 16, 16]               0
            Conv2d-7              [7, 32, 8, 8]           4,640
       BatchNorm2d-8              [7, 32, 8, 8]              64
         LeakyReLU-9              [7, 32, 8, 8]               0
           Conv2d-10              [7, 64, 4, 4]          18,496
      BatchNorm2d-11              [7, 64, 4, 4]             128
        LeakyReLU-12              [7, 64, 4, 4]               0
           Conv2d-13             [7, 128, 2, 2]          73,856
      BatchNorm2d-14             [7, 12

In [5]:
def compute_loss(x, x_new, mu, std):
    kld_weight = 0.00001
    recons_weight = 10

    recons_loss = F.mse_loss(x, x_new) * recons_weight
    kld_loss = (std**2 + mu**2 - torch.log(std**2) - 1/2).sum() * kld_weight

    loss = recons_loss + kld_loss
    return loss, recons_loss.detach(), kld_loss.detach()

In [36]:
model = VAE(512)

optimizer = optim.AdamW(model.parameters(), lr=0.001)

trained_model, metrics = train_model(model, train_loader, val_loader, optimizer, device, num_epochs=60, log_steps=400)

Train preds:


Epoch 0/59, Step 0, Train KL Loss: 1.7002, Train MSE Loss: 4.4158
Val preds:


Epoch 0/59, Step 296, Val KL Loss: 0.1233, Val MSE Loss: 1.4629, Val FID: 4.5130, IS val: 1.0385, IS std: 0.0203
Train preds:


Epoch 1/59, Step 400, Train KL Loss: 0.2052, Train MSE Loss: 1.2397
Val preds:


Epoch 1/59, Step 592, Val KL Loss: 0.1223, Val MSE Loss: 1.0053, Val FID: 2.3297, IS val: 1.0341, IS std: 0.0270
Train preds:


...

Epoch 57/59, Step 17168, Val KL Loss: 0.1243, Val MSE Loss: 0.4954, Val FID: 7.3450, IS val: 1.0218, IS std: 0.0074
Train preds:


Epoch 58/59, Step 17200, Train KL Loss: 0.2057, Train MSE Loss: 0.5595
Val preds:


Epoch 58/59, Step 17464, Val KL Loss: 0.1245, Val MSE Loss: 0.4908, Val FID: 6.7513, IS val: 1.0216, IS std: 0.0100
Train preds:


Epoch 59/59, Step 17600, Train KL Loss: 0.2048, Train MSE Loss: 0.5145
Val preds:


Epoch 59/59, Step 17760, Val KL Loss: 0.1248, Val MSE Loss: 0.4883, Val FID: 7.0624, IS val: 1.0209, IS std: 0.0108


In [37]:
plot_training_metrics(metrics)

/var/tmp/ipykernel_1329026/3267371277.py:31: UserWarning: tight_layout not applied: number of columns in subplot specifications must be multiples of one another.
  plt.tight_layout()


In [6]:
df = pd.read_csv("/home/anmilka/different/list_attr_celeba.csv")[["image_id", "Male"]]
train_genders = []

train_loader = DataLoader(train_dataset,
                            batch_size=batch_size,
                            shuffle=True,
                            num_workers=num_workers,
                            pin_memory=True,
                            drop_last = False)

for path in sorted(train_paths):
    name = Path(path).name
    gender = df.loc[df['image_id'] == name]["Male"].item()
    train_genders.append(gender)

test_genders = []
for path in sorted(test_paths):
    name = Path(path).name
    gender = df.loc[df['image_id'] == name]["Male"].item()
    test_genders.append(gender)

In [7]:
class ImageCondDataset(Dataset):
    def __init__(self, paths, transform, labels):
        self.transform = transform
        self.labels = labels
        self.paths = sorted(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        try:
            image = Image.open(self.paths[index])
        except:
            print("error in open", self.paths[index])

        try:
            image_tensor = self.transform(image)
        except Exception as e:
            print("error in transform", self.paths[index], e)
            return

        label_ch = torch.zeros((1, 64, 64)) + self.labels[index]


        return image_tensor, label_ch, torch.tensor([self.labels[index]])


train_dataset = ImageCondDataset(paths=sorted(train_paths),
                            transform=transforms_,
                            labels=train_genders)
                            
test_dataset = ImageCondDataset(paths=sorted(test_paths),
                            transform=transforms_,
                            labels=test_genders)

print(len(train_dataset), len(test_dataset))

batch_size = 64
num_workers = 2

train_loader = DataLoader(train_dataset,
                            batch_size=batch_size,
                            shuffle=True,
                            num_workers=num_workers,
                            pin_memory=True,
                            drop_last = True)

val_loader = DataLoader(test_dataset,
                            batch_size=batch_size,
                            shuffle=False,
                            num_workers=num_workers,
                            pin_memory=True)

print(len(train_loader), len(val_loader))

18977 999
296 16


In [8]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.fc = nn.Linear(latent_dim+1, hidden_sizes[0] * 1 * 1)
        self.layers = nn.Sequential(
            nn.ConvTranspose2d(hidden_sizes[0], hidden_sizes[1], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[1]),
            nn.LeakyReLU(),

            nn.ConvTranspose2d(hidden_sizes[1], hidden_sizes[2], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[2]),
            nn.LeakyReLU(),

            nn.ConvTranspose2d(hidden_sizes[2], hidden_sizes[3], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[3]),
            nn.LeakyReLU(),

            nn.ConvTranspose2d(hidden_sizes[3], hidden_sizes[4], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[4]),
            nn.LeakyReLU(),

            nn.ConvTranspose2d(hidden_sizes[4], hidden_sizes[5], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[5]),
            nn.LeakyReLU(),

            nn.ConvTranspose2d(hidden_sizes[5], hidden_sizes[6], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_sizes[6]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[6], out_channels=3, kernel_size=3, padding= 1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.fc(x)
        x = x.reshape(-1, hidden_sizes[0], 1, 1)
        x = self.layers(x)
        return x

class Encoder(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(hidden_sizes[-1]+1, hidden_sizes[-2], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_sizes[-2]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[-2], hidden_sizes[-3], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_sizes[-3]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[-3], hidden_sizes[-4], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_sizes[-4]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[-4], hidden_sizes[-5], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_sizes[-5]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[-5], hidden_sizes[-6], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(hidden_sizes[-6]),
            nn.LeakyReLU(),

            nn.Conv2d(hidden_sizes[-6], hidden_sizes[-7], kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU()
        )
        self.fc_mu = nn.Linear(hidden_sizes[0], latent_dim)
        self.fc_var = nn.Linear(hidden_sizes[0], latent_dim)

    def forward(self, x):
        x = self.layers(x)
        x = torch.flatten(x, 1)
    
        mu = self.fc_mu(x)
        std = self.fc_var(x)

        return mu, std

# def reparametrization_trick(mu, std):
#         std = torch.exp(std)
#         eps = torch.randn_like(std)
#         z = eps * std + mu

#         return z

class CVAE(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.encoder = Encoder(latent_dim=latent_dim)
        self.decoder = Decoder(latent_dim=latent_dim)

    def forward(self, x, label_ch, label):
        x_labelled = torch.hstack((label_ch, x))
        mu, std = self.encoder(x_labelled)

        # reparametrization trick
        z = reparametrization_trick(mu, std)

        z_labelled = torch.hstack((label, z))
        x_new = self.decoder(z_labelled)
    
        return x_new, mu, std

m = CVAE(100).to(device)
x = torch.randint(3, 10, (2, 3, 64, 64), dtype=torch.float).to(device)
l_ch = torch.randint(3, 10, (2, 1, 64, 64), dtype=torch.float).to(device)
l = torch.randint(3, 10, (2, 1), dtype=torch.float).to(device)

r = m(x, l_ch, l)

In [9]:
def train_model(model, train_loader, val_loader, optimizer, device, num_epochs=25, log_steps=20):
    model.to(device)
    metrics = {}

    metrics["train_kl"] = []
    metrics["train_mse"] = []

    metrics["val_kl"] = []
    metrics["val_mse"] = []

    metrics["val_fid"] = []
    metrics["is_value"] = []
    metrics["is_std"] = []

    step = 0

    for epoch in range(num_epochs):
        model.train()
        for inputs, labels_ch, labels in train_loader:
            inputs, labels_ch, labels = inputs.to(device), labels_ch.to(device), labels.to(device)
            optimizer.zero_grad()

            preds, mu, std = model(inputs, labels_ch, labels)
            loss, recons_loss, kld_loss = compute_loss(inputs, preds, mu, std)
            if step % log_steps == 0:
                metrics["train_kl"].append(kld_loss.item())
                metrics["train_mse"].append(recons_loss.item())

                print("Train preds:")
                show(vutils.make_grid(preds[:4]))        

                print(f'Epoch {epoch}/{num_epochs - 1}, Step {step}, Train KL Loss: {metrics["train_kl"][-1]:.4f}, Train MSE Loss: {metrics["train_mse"][-1]:.4f}')

            loss.backward()
            optimizer.step()
            step +=1

        # Validation phase
        model.eval()

        with torch.no_grad():
            for inputs, labels_ch, labels in val_loader:
                inputs, labels_ch, labels = inputs.to(device), labels_ch.to(device), labels.to(device)
                optimizer.zero_grad()

                preds, mu, std = model(inputs, labels_ch, labels)
                loss, recons_loss, kld_loss = compute_loss(inputs, preds, mu, std)

                metrics["val_kl"].append(kld_loss.item())
                metrics["val_mse"].append(recons_loss.item())

                fid_metric = FrechetInceptionDistance(feature=64, normalize=True).to(device)
                is_metric = InceptionScore(feature=64, normalize=True).to(device)

                fid_metric.update(inputs, real=True)
                fid_metric.update(preds, real=False)
                is_metric.update(preds)

                fid_value = fid_metric.compute()
                is_value, is_std = is_metric.compute()

                metrics["val_fid"].append(fid_value.item())
                metrics["is_value"].append(is_value.item())
                metrics["is_std"].append(is_std.item())

            print("Val preds:")
            show(vutils.make_grid(preds[:4]))   
     
            print(f'Epoch {epoch}/{num_epochs - 1}, Step {step}, Val KL Loss: {metrics["val_kl"][-1]:.4f}, Val MSE Loss: {metrics["val_mse"][-1]:.4f},\
 Val FID: {metrics["val_fid"][-1]:.4f}, IS val: {metrics["is_value"][-1]:.4f}, IS std: {metrics["is_std"][-1]:.4f}')

    return model, metrics

In [10]:
model = CVAE(512)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
trained_model, metrics = train_model(model, train_loader, val_loader, optimizer, device, num_epochs=60, log_steps=400)

Train preds:


Epoch 0/59, Step 0, Train KL Loss: 1.7156, Train MSE Loss: 5.1354


/home/anmilka/miniconda3/envs/common_latest_libs/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


Val preds:


Epoch 0/59, Step 296, Val KL Loss: 0.1260, Val MSE Loss: 1.6264, Val FID: 4.3651, IS val: 1.0320, IS std: 0.0215
Train preds:


...

Epoch 57/59, Step 17168, Val KL Loss: 0.1255, Val MSE Loss: 0.4911, Val FID: 5.7369, IS val: 1.0211, IS std: 0.0105
Train preds:


Epoch 58/59, Step 17200, Train KL Loss: 0.2072, Train MSE Loss: 0.5431
Val preds:


Epoch 58/59, Step 17464, Val KL Loss: 0.1257, Val MSE Loss: 0.4725, Val FID: 5.1223, IS val: 1.0235, IS std: 0.0091
Train preds:


Epoch 59/59, Step 17600, Train KL Loss: 0.2045, Train MSE Loss: 0.4752
Val preds:


Epoch 59/59, Step 17760, Val KL Loss: 0.1246, Val MSE Loss: 0.4837, Val FID: 5.3212, IS val: 1.0205, IS std: 0.0099


In [11]:
plot_training_metrics(metrics)

/var/tmp/ipykernel_95960/3267371277.py:31: UserWarning: tight_layout not applied: number of columns in subplot specifications must be multiples of one another.
  plt.tight_layout()


In [43]:
label = torch.zeros([24, 1]) + 1
label = label.to(device)
z = torch.randn(24, 512)
z = z.to(device)

z_labelled = torch.hstack((label, z))

preds = model.decoder(z_labelled)
show(vutils.make_grid(preds))

In [44]:
label = torch.zeros([24, 1]) - 1
label = label.to(device)
z = torch.randn(24, 512)
z = z.to(device)

z_labelled = torch.hstack((label, z))

preds = model.decoder(z_labelled)
show(vutils.make_grid(preds))